# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shamiquekhan/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring (Lane 2)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup: navigate to repo root, import, load data
import os, sys, warnings, json
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shamiquekhan/flyrank-ml-internship", "flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print(f"Working dir: {os.getcwd()}")

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")

# Target
df['target'] = (df['trend_direction'].str.lower() == 'down').astype(int)
print(f"Declining rate: {df['target'].mean():.3f}")

# Load model predictions from Week 5
preds = pd.read_csv('work/outputs/model_predictions.csv')
print(f"Model predictions: {len(preds):,} rows")

# Load baseline metrics
with open('work/outputs/baseline_metrics.json') as f:
    baseline_metrics = json.load(f)

# Merge predictions with full dataframe for richer context
queue = df.merge(preds[['content_id', 'model_score', 'model_pred']], on='content_id', how='inner')
print(f"Merged queue: {len(queue):,} rows")

Working dir: /home/shamique/flyrank/ml1/flyrank-ml-internship
Loaded 30,000 rows, 44 columns
Declining rate: 0.542
Model predictions: 3,980 rows
Merged queue: 3,980 rows


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Reason code taxonomy (extends Week-4 baseline)

The model outputs a probability score (0–1). We translate this into **action labels** with **reason codes** that explain *why* a page ranks where it does. This mirrors the baseline's reason codes but adds model-driven nuance.

| Reason Code | Action Label | When It Applies | Human Meaning |
|---|---|---|---|
| `HIGH_DECLINE_RISK` | Refresh Content | `model_score >= 0.7` AND `target == 1` (or strong signals) | Model confidently flags declining page with high impressions — priority review |
| `STALE_HIGH_VISIBILITY` | Refresh Content | `days_since_last_update >= 180` AND `impressions_90d >= 1000` AND `model_score >= 0.5` | Old page still earning traffic — refresh before decay |
| `CTR_GAP_HIGH_IMP` | Update Metadata | `model_score >= 0.5` AND `ctr` below position-tier expected AND `impressions_90d >= 500` | Good position but weak CTR — title/meta rewrite candidate |
| `LOW_VISIBILITY_DECLINE` | Monitor | `model_score >= 0.5` AND `impressions_90d < 100` | Declining but negligible traffic — not worth review effort |
| `FALSE_POSITIVE_RISK` | Human Review Required | `model_score >= 0.7` AND `target == 0` (in validation) | Model confident but validation says not declining — check before acting |
| `MONITOR` | Monitor | `model_score < 0.5` | Low risk — no action needed now |

In [2]:
# Define expected CTR by position tier (from baseline)
def expected_ctr_for_pos(pos):
    if pos <= 0 or pd.isna(pos):
        return 0.21
    if pos <= 3:
        return 2.71
    if pos <= 10:
        return 0.65
    if pos <= 20:
        return 0.32
    return 0.21

queue['expected_ctr'] = queue['avg_position'].apply(expected_ctr_for_pos)
queue['ctr_gap'] = (queue['expected_ctr'] - queue['ctr']).clip(0)

# Assign reason codes
def assign_reason(row):
    score = row['model_score']
    impressions = row['impressions_90d']
    stale_days = row['days_since_last_update']
    ctr_gap = row['ctr_gap']
    target = row['target']
    
    # High confidence declining + high impressions = top priority
    if score >= 0.7 and impressions >= 1000:
        return 'HIGH_DECLINE_RISK'
    # Stale + visible
    if stale_days >= 180 and impressions >= 1000 and score >= 0.5:
        return 'STALE_HIGH_VISIBILITY'
    # CTR gap with meaningful impressions
    if ctr_gap > 0.2 and impressions >= 500 and score >= 0.5:
        return 'CTR_GAP_HIGH_IMP'
    # Declining but low visibility
    if score >= 0.5 and impressions < 100:
        return 'LOW_VISIBILITY_DECLINE'
    # Model says declining but validation disagrees (FP risk)
    # Note: we don't have target in production; this is for playbook transparency
    if score >= 0.7:
        return 'HIGH_DECLINE_RISK'
    # Default
    return 'MONITOR'

def assign_action(reason):
    mapping = {
        'HIGH_DECLINE_RISK': 'Refresh Content',
        'STALE_HIGH_VISIBILITY': 'Refresh Content',
        'CTR_GAP_HIGH_IMP': 'Update Metadata',
        'LOW_VISIBILITY_DECLINE': 'Monitor',
        'FALSE_POSITIVE_RISK': 'Human Review Required',
        'MONITOR': 'Monitor'
    }
    return mapping.get(reason, 'Monitor')

queue['reason_code'] = queue.apply(assign_reason, axis=1)
queue['action_label'] = queue['reason_code'].apply(assign_action)

# Rank by model_score descending
queue = queue.sort_values('model_score', ascending=False).reset_index(drop=True)
queue['rank'] = range(1, len(queue) + 1)

print("=== Reason code distribution ===")
print(queue['reason_code'].value_counts().to_string())
print()
print("=== Action label distribution ===")
print(queue['action_label'].value_counts().to_string())

=== Reason code distribution ===
reason_code
MONITOR                   2713
CTR_GAP_HIGH_IMP           632
HIGH_DECLINE_RISK          324
LOW_VISIBILITY_DECLINE     311

=== Action label distribution ===
action_label
Monitor            3024
Update Metadata     632
Refresh Content     324


In [3]:
# Top 20 ranked queue for human review
top20 = queue.head(20)[[
    'rank', 'content_id', 'client_id', 'model_score', 'reason_code', 'action_label',
    'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update',
    'content_age_days', 'word_count', 'target'
]].copy()

print("=== TOP 20 RANKED QUEUE ===")
for _, row in top20.iterrows():
    print(f"Rank {int(row['rank']):3d} | Score: {row['model_score']:.3f} | {row['reason_code']:25s} | {row['action_label']:20s} |")
    print(f"  ID: {row['content_id']} | Client: {row['client_id']} | Imp: {row['impressions_90d']:,.0f} | Pos: {row['avg_position']:.1f} | CTR: {row['ctr']:.2f} | Stale: {row['days_since_last_update']:.0f}d | Age: {row['content_age_days']:.0f}d | Words: {row['word_count']:,.0f} | Actual declining: {int(row['target'])}")
    print()

=== TOP 20 RANKED QUEUE ===
Rank   1 | Score: 0.854 | HIGH_DECLINE_RISK         | Refresh Content      |
  ID: content_55f2721bcdc2 | Client: client_3fdba35f04 | Imp: 2,294 | Pos: 35.9 | CTR: 0.00 | Stale: 104d | Age: 284d | Words: 1,380 | Actual declining: 1

Rank   2 | Score: 0.845 | HIGH_DECLINE_RISK         | Refresh Content      |
  ID: content_d9c30b1d8907 | Client: client_7f2253d7e2 | Imp: 18,384 | Pos: 38.6 | CTR: 0.03 | Stale: 20d | Age: 132d | Words: 4,473 | Actual declining: 1

Rank   3 | Score: 0.842 | HIGH_DECLINE_RISK         | Refresh Content      |
  ID: content_86444e331502 | Client: client_3fdba35f04 | Imp: 462 | Pos: 24.7 | CTR: 0.00 | Stale: 104d | Age: 310d | Words: 1,373 | Actual declining: 1

Rank   4 | Score: 0.841 | HIGH_DECLINE_RISK         | Refresh Content      |
  ID: content_d5755a0962c8 | Client: client_7f2253d7e2 | Imp: 5,566 | Pos: 24.1 | CTR: 0.05 | Stale: 20d | Age: 147d | Words: 3,990 | Actual declining: 1

Rank   5 | Score: 0.839 | HIGH_DECLINE_RISK

In [4]:
# Summary stats by reason code
summary = queue.groupby('reason_code').agg(
    count=('content_id', 'count'),
    avg_score=('model_score', 'mean'),
    avg_impressions=('impressions_90d', 'mean'),
    avg_position=('avg_position', 'mean'),
    avg_ctr=('ctr', 'mean'),
    avg_stale_days=('days_since_last_update', 'mean'),
    declining_rate=('target', 'mean')
).round(2)

print("=== REASON CODE SUMMARY ===")
print(summary.to_string())

# Export ranked queue for paper
output_cols = ['rank', 'content_id', 'client_id', 'model_score', 'reason_code', 'action_label',
               'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update',
               'content_age_days', 'word_count', 'target']
queue_output = queue[output_cols].copy()

os.makedirs('work/outputs', exist_ok=True)
queue_output.to_csv('work/outputs/action_playbook_queue.csv', index=False)
print(f"\nExported ranked queue to work/outputs/action_playbook_queue.csv ({len(queue_output)} rows)")

=== REASON CODE SUMMARY ===
                        count  avg_score  avg_impressions  avg_position  avg_ctr  avg_stale_days  declining_rate
reason_code                                                                                                     
CTR_GAP_HIGH_IMP          632       0.63          4336.54         12.09     0.12           41.01            0.75
HIGH_DECLINE_RISK         324       0.75          5798.56         19.34     0.13           67.09            0.93
LOW_VISIBILITY_DECLINE    311       0.55            49.88         16.90     0.13           36.51            0.67
MONITOR                  2713       0.46          7042.55         17.24     0.40           41.31            0.52

Exported ranked queue to work/outputs/action_playbook_queue.csv (3980 rows)


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use

**Who:** Content operations specialists, SEO strategists, editorial leads at FlyRank or client teams.

**What:** A **prioritization aid** — a ranked queue of pages to review for content refresh, metadata updates, or monitoring. The model score and reason codes help humans decide *where to look first* given limited reviewer capacity.

**How it fits the workflow:**
1. Content team pulls the weekly/monthly ranked queue
2. They review the top-N pages (e.g., top 50) in order
3. For each page, the reason code tells them *what to check*: content freshness, title/meta, or just monitor
4. They apply human judgment: competitive landscape, business priority, resource constraints
5. They take action (refresh, rewrite, no-op) and log the outcome

**Decision-support framing:** The model **ranks** pages by estimated decline risk; it does not **decide** what action to take. The human reviewer is the decision-maker; the model is the flashlight.

### Limits (Where This Stops Being Valid)

| Limit | Explanation |
|---|---|
| **Single snapshot** | Trained on one 90-day window (30K pages, 32 clients). No temporal generalization tested. |
| **Proxy label** | Target = current trend direction (last-30d vs prev-30d), not future decline. A page "declining now" may recover naturally. |
| **Grouped split drop** | Precision@50 drops from 96% (random) to ~74% (client-grouped). Model memorizes client patterns; generalization to new clients is unproven. |
| **No causal claim** | Refreshing a flagged page may not reverse decline. The model identifies *correlates* of current decline, not levers that cause recovery. |
| **Feature gaps** | No trend features (impression_trend, ctr_trend), no query-level signals, no backlink data. Sudden drops on fresh pages are missed. |
| **Starter dataset only** | 30K rows is a sample. Warehouse has 79M daily rows across 57 brands — patterns may differ. |
| **No cost model** | Refresh effort vs. expected uplift not quantified. High-impression pages may be expensive to refresh. |
| **Client heterogeneity** | 32 clients with 3–7008 rows each. Model may overfit large clients and underfit small ones. |

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Checklist (per page)

Before taking any action on a flagged page, the reviewer **must** verify:

1. **Business context** — Is this page strategically important? (e.g., flagship product, high-revenue keyword, brand term)
2. **Competitive landscape** — Has a competitor published something better recently? SERP changed?
3. **Content quality** — Is the page actually thin/outdated, or is the model picking up a false signal? (e.g., seasonal dip, tracking issue)
4. **Refresh cost** — Will a meaningful update take 30 minutes or 3 days? Is the ROI there?
5. **Cannibalization check** — Are multiple pages targeting the same intent? Consolidation may beat refresh.
6. **Technical health** — Is the page indexable, mobile-friendly, fast? Fix technical issues before content.
7. **Historical pattern** — Has this page been refreshed before? Did it help? (Avoid repeat low-ROI refreshes)

### The No-Go List (What Should NEVER Be Automated)

| Action | Why It Must Stay Human |
|---|---|
| **Auto-refresh content** | Refreshing without editorial judgment produces filler. Quality depth beats arbitrary length (Finding #21 nuance). |
| **Auto-rewrite titles/metas** | CTR optimization needs intent understanding. Generic templates hurt more than help. |
| **Auto-redirect or delete pages** | A declining page may have brand value, backlinks, or assist conversions. Deletion is a strategic call. |
| **Allocate budget based solely on score** | High score ≠ high ROI. Low-visibility declining pages (LOW_VISIBILITY_DECLINE) score high but aren't worth the effort. |
| **Trigger recrawl automatically** | Recrawl budget is finite. Prioritize pages where changes are substantial and measurable. |
| **Replace human QA** | Model false positives exist (FP rate ~20% in grouped split). Every top-N item needs human eyes. |
| **Generalize to new clients/verticals** | Model trained on 32 clients in one snapshot. New verticals need re-validation. |

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring Triggers (Run These Checks Monthly)

| Metric / Signal | Threshold | Action |
|---|---|---|
| **Queue base rate drift** | `declining_rate` in new data shifts >10pp from training (54.2%) | Retrain — label distribution shifted |
| **Precision@50 on recent labels** | Measured P@50 on last month's outcomes drops below baseline (58%) | Retrain — model no longer beats rule |
| **Feature distribution shift** | KS-test on top 5 features (impressions, age, position, CTR, stale_days) p < 0.01 | Investigate data pipeline or retrain |
| **Client mix change** | New clients >20% of queue, or top-3 clients drop >50% share | Retrain with new client data |
| **Action outcome tracking** | Of pages acted on, <30% show impression improvement at 60 days | Review reason codes / human process |
| **False positive rate** | Human reviewers mark >40% of top-50 as "no action needed" | Adjust threshold or add features |

### Retrain Triggers (When to Rebuild the Model)

1. **Scheduled:** Quarterly retrain on latest 90-day snapshot (minimum)
2. **Event-driven:** Major Google algorithm update (core update, helpful content, etc.)
3. **Performance-driven:** Any monitoring threshold breached
4. **Data-driven:** Warehouse panel expands (new clients, longer history) — retrain on full panel with time-aware split

### Light Monitoring Dashboard (What to Track Weekly)

- Queue size by reason code (are we getting more HIGH_DECLINE_RISK?)
- Top-50 review completion rate (are humans keeping up?)
- Action taken distribution (Refresh vs Metadata vs Monitor)
- 30-day impression change for acted-on pages (leading indicator of value)

In [7]:
# Export monitoring thresholds as JSON for paper
monitoring_config = {
    "base_rate_training": 0.542,
    "baseline_precision_at_50": 0.58,
    "model_precision_at_50_random": 0.96,
    "model_precision_at_50_grouped": 0.744,
    "drift_thresholds": {
        "base_rate_shift_pp": 10,
        "precision_at_50_min": 0.58,
        "ks_test_pvalue": 0.01,
        "new_client_share_max": 0.20,
        "action_success_rate_min": 0.30,
        "human_fp_rate_max": 0.40
    },
    "retrain_cadence": "quarterly_minimum",
    "event_triggers": ["google_core_update", "major_traffic_shift", "warehouse_panel_expansion"],
    "weekly_dashboard_metrics": [
        "queue_size_by_reason_code",
        "top50_review_completion_rate",
        "action_taken_distribution",
        "acted_pages_30d_impression_change"
    ]
}

with open('work/outputs/monitoring_config.json', 'w') as f:
    json.dump(monitoring_config, f, indent=2)
print("Exported monitoring_config.json")

Exported monitoring_config.json


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [8]:
# Export 1: Full ranked queue (already saved above)
# Export 2: Top-N for paper figures
top50 = queue.head(50)[output_cols].copy()
top50.to_csv('work/outputs/action_playbook_top50.csv', index=False)
print(f"Exported top50: {len(top50)} rows")

# Export 3: Reason code summary for paper table
summary.to_csv('work/outputs/action_playbook_reason_summary.csv')
print(f"Exported reason summary: {len(summary)} rows")

# Export 4: Key metrics for paper (JSON receipt)
paper_metrics = {
    "queue_total_pages": len(queue),
    "model_used": "RandomForest (Week-5, grouped-split validated)",
    "base_rate": baseline_metrics['precision_at_k']['base_rate'],
    "baseline_precision_at_50": baseline_metrics['precision_at_k']['p50'],
    "model_precision_at_50_random": 0.96,
    "model_precision_at_50_grouped": 0.744,
    "reason_code_counts": queue['reason_code'].value_counts().to_dict(),
    "action_label_counts": queue['action_label'].value_counts().to_dict(),
    "top50_declining_rate": float(top50['target'].mean()),
    "limits": [
        "Single 90-day snapshot (no temporal generalization)",
        "Proxy label: current trend direction, not future decline",
        "Grouped split P@50 = 74.4% (random split overestimates)",
        "No causal claim: refresh may not reverse decline",
        "Starter dataset only (30K rows, 32 clients)"
    ],
    "no_go_automation": [
        "Auto-refresh content",
        "Auto-rewrite titles/metas",
        "Auto-redirect or delete pages",
        "Allocate budget based solely on score",
        "Trigger recrawl automatically",
        "Replace human QA",
        "Generalize to new clients/verticals without re-validation"
    ]
}

with open('work/outputs/action_playbook_paper_metrics.json', 'w') as f:
    json.dump(paper_metrics, f, indent=2)
print("Exported action_playbook_paper_metrics.json")

# Export 5: Cost/value thinking template (for paper discussion)
cost_value = {
    "refresh_cost_estimate_hours": {
        "light_touch": 0.5,
        "moderate_expand": 2.0,
        "full_rewrite": 8.0
    },
    "expected_uplift_scenarios": {
        "high_impression_stale": {"impressions_gain_pct": 50, "confidence": "directional"},
        "ctr_gap_page1": {"ctr_lift_pct": 30, "confidence": "directional"},
        "low_visibility": {"impressions_gain_pct": 10, "confidence": "low"}
    },
    "decision_rule": "Only act on HIGH_DECLINE_RISK or STALE_HIGH_VISIBILITY with impressions >= 1000 AND human confirms content quality issue. CTR_GAP_HIGH_IMP gets metadata review first (lower cost). LOW_VISIBILITY_DECLINE = monitor only."
}

with open('work/outputs/action_playbook_cost_value.json', 'w') as f:
    json.dump(cost_value, f, indent=2)
print("Exported action_playbook_cost_value.json")

print("\n=== ALL EXPORTS COMPLETE ===")
print("Files in work/outputs/:\n")
import os
for f in sorted(os.listdir('work/outputs')):
    if f.startswith('action_playbook') or f.startswith('monitoring'):
        print(f"  {f}")

Exported top50: 50 rows
Exported reason summary: 4 rows
Exported action_playbook_paper_metrics.json
Exported action_playbook_cost_value.json

=== ALL EXPORTS COMPLETE ===
Files in work/outputs/:

  action_playbook_cost_value.json
  action_playbook_paper_metrics.json
  action_playbook_queue.csv
  action_playbook_reason_summary.csv
  action_playbook_top50.csv
  monitoring_config.json


---
## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] **Ranked actions + reason codes** with human-readable explanations
- [x] **Intended use and limits** clearly stated (proxy label, snapshot, grouped split drop, no causal claim)
- [x] **Human review checklist + no-go list** for automation boundaries
- [x] **Monitoring/retrain triggers** with specific thresholds
- [x] **Exports for paper**: queue CSV, top50 CSV, reason summary CSV, paper metrics JSON, cost/value JSON, monitoring config JSON
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.